# DQN Experiment Guide

This notebook provides a code-only workflow for comparing DQN, Double DQN, and Dueling DQN on Atari Breakout. It uses grayscale 84x84 observations with four-frame stacks, writes TensorBoard logs, and saves checkpoints. The notebook is intentionally not executed before committing.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

import ale_py
import gymnasium as gym
import numpy as np
import torch
from gymnasium.wrappers import AtariPreprocessing, TimeLimit

gym.register_envs(ale_py)

DEEP_RL_DIR = Path.cwd()
if DEEP_RL_DIR.name != 'deep-rl':
    DEEP_RL_DIR = Path('/home/dangineer/Learnspaces/coursera/reinforcement-learning-specialization/rl-lab/deep-rl')
if str(DEEP_RL_DIR) not in sys.path:
    sys.path.insert(0, str(DEEP_RL_DIR))

from dqns.dqn import DQNAgent
from dqns.double_dqn import DoubleDQNAgent
from dqns.dueling_dqn import DuelingDQNAgent

## 1. Configure the experiment

Use a small TRAINING_STEPS value while debugging. Full Atari runs are computationally expensive and should be launched deliberately.

In [ ]:
ENV_ID = 'ALE/Breakout-v5'
GLOBAL_SEED = 2026
NUM_SEEDS = 3
TRAINING_STEPS = 1_000_000
EXPLORATION_STEPS = 250_000
REPLAY_BUFFER_SIZE = 100_000
BATCH_SIZE = 64
LEARNING_STARTS = 50_000
TRAIN_FREQ = 4
TARGET_UPDATE_FREQ = 5_000
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LOG_DIR = DEEP_RL_DIR / 'runs' / 'dqns_comparison'
CHECKPOINT_DIR = LOG_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

seed_generator = np.random.default_rng(GLOBAL_SEED)
SEEDS = [int(seed) for seed in seed_generator.integers(0, 2**31 - 1, size=NUM_SEEDS)]

print(f'Environment: {ENV_ID}')
print(f'Device: {DEVICE}')
print(f'Experiment seeds: {SEEDS}')
print(f'Training steps: {TRAINING_STEPS}')
print(f'Replay buffer size: {REPLAY_BUFFER_SIZE}')

## 2. Create the preprocessed Atari environment

The agent expects uint8 grayscale frames with shape (84, 84). The replay buffer constructs four-frame stacks for the convolutional network. A life loss ends a training episode, matching the experiment setup used by the agents.

In [ ]:
class LifeLossWrapper(gym.Wrapper):
    """End a training episode when Breakout loses one life."""

    def __init__(self, env):
        super().__init__(env)
        self._lives = 0

    def reset(self, **kwargs):
        observation, info = self.env.reset(**kwargs)
        self._lives = int(self.unwrapped.ale.lives())
        return observation, info

    def step(self, action):
        observation, reward, terminated, truncated, info = self.env.step(action)
        lives = int(self.unwrapped.ale.lives())
        life_lost = lives < self._lives
        self._lives = lives
        if life_lost and not terminated and not truncated:
            terminated = True
            info = dict(info)
            info['life_lost'] = True
        return observation, reward, terminated, truncated, info


def make_env(seed, training_mode=True):
    env = gym.make(ENV_ID, frameskip=1)
    env = AtariPreprocessing(
        env,
        frame_skip=4,
        screen_size=84,
        grayscale_obs=True,
        scale_obs=False,
    )
    if training_mode:
        env = LifeLossWrapper(env)
    env = TimeLimit(env, max_episode_steps=10_000)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    return env

test_env = make_env(SEEDS[0])
state, _ = test_env.reset(seed=SEEDS[0])
assert state.shape == (84, 84)
assert state.dtype == np.uint8
print(f'Observation shape: {state.shape}')
print(f'Number of actions: {test_env.action_space.n}')
test_env.close()

## 3. Define the agents and checkpoints

Each agent uses the same environment and training budget. The algorithm-specific difference is isolated in the agent implementation: DQN uses the target network for both selection and evaluation, Double DQN separates those operations, and Dueling DQN changes the network representation.

In [ ]:
agent_specs = {
    'dqn': DQNAgent,
    'double_dqn': DoubleDQNAgent,
    'dueling_dqn': DuelingDQNAgent,
}

checkpoint_paths = {
    name: {seed: CHECKPOINT_DIR / f'{name}_seed_{seed}.pt' for seed in SEEDS}
    for name in agent_specs
}

print(f'Planned runs: {len(agent_specs) * len(SEEDS)}')

## 4. Run one configured agent

Select one entry from agent_specs and run this cell. Repeat for the other variants when you want the complete comparison. Checkpoints contain both online and target network weights.

In [ ]:
agent_name = 'dqn'
agent_type = agent_specs[agent_name]

for seed in SEEDS:
    train_env = make_env(seed)
    agent = agent_type(
        action_dim=train_env.action_space.n,
        device=DEVICE,
        seed=seed,
        exploration_steps=EXPLORATION_STEPS,
        buffer_size=REPLAY_BUFFER_SIZE,
        batch_size=BATCH_SIZE,
        learning_starts=LEARNING_STARTS,
        train_freq=TRAIN_FREQ,
        target_update_freq=TARGET_UPDATE_FREQ,
        tensorboard_log_dir=str(LOG_DIR),
        run_name=agent_name,
    )
    agent.learn(train_env, max_steps=TRAINING_STEPS)
    torch.save(
        {
            'agent_name': agent_name,
            'seed': seed,
            'action_dim': train_env.action_space.n,
            'network_state_dict': agent.network.state_dict(),
            'target_network_state_dict': agent.target_network.state_dict(),
        },
        checkpoint_paths[agent_name][seed],
    )
    agent.close()
    train_env.close()
    print(f'{agent_name}, seed={seed}: training and checkpoint saved')

## 5. Inspect logs with TensorBoard

Run this after training one or more variants. Compare episode returns, losses, epsilon, and Q-value statistics across runs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/dqns_comparison

## 6. Hyperparameter tuning

**DQN and its variants, Double DQN and Dueling DQN, have some important hyperparameters that we need to choose carefully to make the learning process more effective:**

- **Learning rate:** We need to tune this param in quite many RL algorithms, it controls how quickly the network updates its weights. A learning rate that is too large can make training unstable, while a very small value can make training too slow.

- **Discount factor ($\gamma$):** Controls how much the agent cares about future rewards. A value close to `1` makes the agent consider long-term rewards.

- **Epsilon values:** `epsilon_start` controls the initial exploration level, while `epsilon_end` controls the minimum exploration level. The agent usually starts by exploring more and gradually becomes more greedy.

- **Exploration steps:** Controls how quickly epsilon decreases. If epsilon decreases too quickly, the agent may stop exploring before it has learned enough.

- **Replay buffer size:** Determines how many past transitions are stored. A larger buffer provides more diverse experiences, but it also uses more memory.

- **Batch size:** Determines how many transitions are used for each update. Larger batches can make learning more stable, but they require more memory and computation.

- **Training frequency:** Defines how often the online network is updated. Updating too often can be expensive, while updating too rarely can slow down learning.

- **Target update frequency:** Controls how often the target network is updated. A target network that changes too often can make training unstable.

- **Network architecture:** The network should be large enough to learn the task, but not unnecessarily complex. Dueling DQN uses a different architecture by separating the value and advantage streams.

In practice, we should start with simple default values and change only a few hyperparameters at a time. This makes it easier to understand which hyperparameter affects the learning process.